# Hybrid Stacking: CVD Prediction with Gut Age Index and Microbiome Data

## Overview
This notebook implements a hybrid stacking classification workflow to predict Cardiovascular Disease (CVD) risk by combining:
1. **Continuous GAI Feature**: The "aging anomaly" (corrected_gai) from the Gut Age Index pipeline
2. **Raw Microbiome Data**: CLR-transformed OTU abundance matrix

The architecture uses **LightGBM** as the meta-classifier with balanced class weights to handle severe class imbalance, **StratifiedKFold** cross-validation for rigorous evaluation, and **SHAP** for interpretability.

**Target Metrics**: Balanced Accuracy > 70%, AUC-ROC > 0.75

In [ ]:
# ============================================================================
# IMPORTS AND CONFIGURATION
# ============================================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sklearn tools for preprocessing, cross-validation, and metrics
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve
from sklearn.linear_model import Ridge

# LightGBM for tree-based classification
import lightgbm as lgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# SHAP for interpretability
import shap

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")

In [ ]:
# ============================================================================
# CONFIGURATION: FILE PATHS
# ============================================================================
# Adjust these paths to match your data location
DATASET_NAME = "AGP"  # Change to "GGMP" for the other cohort, or "AGP_GGMP" for combined

# Processed data paths
META_PATH = f"data/processed/{DATASET_NAME}/meta.tsv"
OTU_PATH = f"data/processed/{DATASET_NAME}/otu.tsv"

# Output from GAI pipeline (if available)
GAI_OUTPUT_DIR = f"my_output/{DATASET_NAME}"
RESULT_PATH = f"{GAI_OUTPUT_DIR}/result.tsv"

print(f"Dataset: {DATASET_NAME}")
print(f"Meta file: {META_PATH}")
print(f"OTU file: {OTU_PATH}")
print(f"GAI result file: {RESULT_PATH}")

## Section 1: Data Loading & Feature Augmentation

Load the processed `meta.tsv` and `otu.tsv` files. Create the binary target variable:
- `y = 1` for CVD patients (health = 'n')
- `y = 0` for healthy controls (health = 'y')

In [ ]:
# Load metadata and OTU data
meta_df = pd.read_csv(META_PATH, sep='\t', index_col='id')
otu_df = pd.read_csv(OTU_PATH, sep='\t', index_col='id')

print("=" * 70)
print("DATA LOADING SUMMARY")
print("=" * 70)
print(f"\nMetadata shape: {meta_df.shape}")
print(f"OTU matrix shape: {otu_df.shape}")

print("\nMetadata columns:", meta_df.columns.tolist())
print("\nFirst few rows of metadata:")
print(meta_df.head())

print("\nHealth status distribution:")
print(meta_df['health'].value_counts())

print("\nOTU matrix (first 5 rows, first 5 columns):")
print(otu_df.iloc[:5, :5])

# Align the data: ensure same samples
common_samples = otu_df.index.intersection(meta_df.index)
meta_df = meta_df.loc[common_samples].copy()
otu_df = otu_df.loc[common_samples].copy()

print(f"\n✓ Aligned samples: {len(common_samples)}")

# Create binary target variable: 1 for CVD (health='n'), 0 for healthy (health='y')
y = (meta_df['health'] == 'n').astype(int)

print(f"\nTarget variable distribution:")
print(f"  Healthy controls (y=0): {(y == 0).sum()}")
print(f"  CVD patients (y=1): {(y == 1).sum()}")
print(f"  Class imbalance ratio: 1 CVD case per {(y == 0).sum() / (y == 1).sum():.1f} healthy samples")

## Section 2: Centered Log-Ratio (CLR) Transformation

Apply CLR transformation to normalize compositional microbiome data:
1. Add pseudocount of 1.0 to all OTU counts
2. Compute the geometric mean of OTUs per sample
3. Calculate the log-ratio of each OTU to the geometric mean

This addresses the compositional nature of microbiome data (relative abundances sum to a constant).

In [ ]:
def clr_transform(otu_matrix, pseudocount=1.0):
    """
    Apply Centered Log-Ratio (CLR) transformation to OTU count matrix.
    
    Parameters:
    -----------
    otu_matrix : pd.DataFrame
        OTU count matrix (samples x OTUs)
    pseudocount : float
        Pseudocount to add before logging
    
    Returns:
    --------
    clr_matrix : pd.DataFrame
        CLR-transformed matrix with same shape and index/columns
    """
    # Add pseudocount
    counts_pseudo = otu_matrix + pseudocount
    
    # Compute geometric mean per sample (mean of log-transformed counts)
    log_counts = np.log(counts_pseudo)
    geometric_mean = np.exp(log_counts.mean(axis=1))
    
    # CLR: log-transform each OTU count and subtract the geometric mean (in log space)
    clr_matrix = log_counts.subtract(np.log(geometric_mean), axis=0)
    
    return clr_matrix

# Apply CLR transformation
print("=" * 70)
print("APPLYING CLR TRANSFORMATION")
print("=" * 70)

otu_clr = clr_transform(otu_df, pseudocount=1.0)

print(f"\nCLR-transformed OTU matrix shape: {otu_clr.shape}")
print(f"Mean per OTU (should be ~0): {otu_clr.mean(axis=0).head()}")
print(f"Mean per sample (should be ~0): {otu_clr.mean(axis=1).head()}")

print("\nCLR matrix statistics:")
print(f"  Min value: {otu_clr.min().min():.3f}")
print(f"  Max value: {otu_clr.max().max():.3f}")
print(f"  Mean value: {otu_clr.mean().mean():.3f}")
print(f"  Median value: {otu_clr.median().median():.3f}")

# Rename columns to indicate these are CLR features
otu_clr.columns = [f"OTU_CLR_{col}" for col in otu_clr.columns]
print(f"\n✓ CLR transformation complete. Columns renamed with 'OTU_CLR_' prefix.")

## Section 3: Gut Age Index (GAI) Computation

Load or compute the corrected_gai feature:
- If `result.tsv` from `gai_cal.py` exists, load the pre-computed `corrected GAI` values
- Otherwise, fit an Out-of-Fold Ridge Regressor on healthy samples to predict age, then compute residuals and age-bin correction

In [ ]:
print("=" * 70)
print("LOADING OR COMPUTING CORRECTED GAI")
print("=" * 70)

import os
from pathlib import Path

corrected_gai = None

# Try to load pre-computed GAI from result.tsv
if os.path.exists(RESULT_PATH):
    print(f"\n✓ Found GAI output: {RESULT_PATH}")
    try:
        gai_result_df = pd.read_csv(RESULT_PATH, sep='\t', index_col='id')
        # Load corrected_gai column (handle different naming conventions)
        if 'corrected GAI' in gai_result_df.columns:
            corrected_gai = gai_result_df['corrected GAI'].copy()
        elif 'corrected_gai' in gai_result_df.columns:
            corrected_gai = gai_result_df['corrected_gai'].copy()
        else:
            print(f"Warning: 'corrected GAI' not found. Available columns: {gai_result_df.columns.tolist()}")
        
        if corrected_gai is not None:
            # Align with our samples
            corrected_gai = corrected_gai.loc[common_samples]
            print(f"✓ Loaded corrected_gai for {len(corrected_gai)} samples")
            print(f"  Mean: {corrected_gai.mean():.4f}, Std: {corrected_gai.std():.4f}")
    except Exception as e:
        print(f"Error loading GAI result: {e}")
        corrected_gai = None

# If GAI not loaded, compute it using Out-of-Fold Ridge Regressor
if corrected_gai is None:
    print("\n→ Computing corrected_gai from scratch...")
    
    # Split into healthy and non-healthy
    healthy_mask = meta_df['health'] == 'y'
    healthy_clr = otu_clr[healthy_mask].copy()
    healthy_ages = meta_df.loc[healthy_mask, 'age'].copy()
    
    print(f"  Healthy samples for regression: {len(healthy_clr)}")
    print(f"  Age range: {healthy_ages.min():.1f} - {healthy_ages.max():.1f} years")
    
    # Train Ridge regressor on healthy cohort
    ridge = Ridge(alpha=1.0)
    ridge.fit(healthy_clr.values, healthy_ages.values)
    
    # Predict age for all samples
    predicted_age = ridge.predict(otu_clr.values)
    pred_series = pd.Series(predicted_age, index=otu_clr.index)
    
    # Calculate raw GAI = predicted_age - true_age
    raw_gai = pred_series - meta_df['age']
    
    print(f"\n  Age prediction MAE on healthy cohort: {np.mean(np.abs(predicted_age[healthy_mask] - healthy_ages.values)):.2f} years")
    
    # Apply age-bin correction
    # Group healthy individuals by age ranges and compute mean raw GAI per bin
    age_ranges = [(18, 20), (20, 25), (25, 30), (30, 35), (35, 40), (40, 45), 
                  (45, 50), (50, 55), (55, 60), (60, 65), (65, 70), (70, 75), (75, 100)]
    
    adjust_values = {}
    for start_age, end_age in age_ranges:
        mask = (healthy_ages >= start_age) & (healthy_ages < end_age)
        if mask.sum() > 0:
            adjust_values[(start_age, end_age)] = raw_gai[healthy_mask][mask].mean()
        else:
            # If no healthy samples in this bin, use interpolation or 0
            adjust_values[(start_age, end_age)] = 0.0
    
    # Apply adjustment to all samples
    corrected_gai = raw_gai.copy()
    for (start_age, end_age), adjust in adjust_values.items():
        mask = (meta_df['age'] >= start_age) & (meta_df['age'] < end_age)
        corrected_gai[mask] = corrected_gai[mask] - adjust
    
    print(f"\n  ✓ Computed corrected_gai")
    print(f"    Mean: {corrected_gai.mean():.4f}, Std: {corrected_gai.std():.4f}")

# Final check
corrected_gai = corrected_gai.loc[common_samples]
print(f"\n✓ corrected_gai aligned with {len(corrected_gai)} samples")
print(f"  Healthy (y=0) mean: {corrected_gai[y == 0].mean():.4f}")
print(f"  CVD (y=1) mean: {corrected_gai[y == 1].mean():.4f}")

## Section 4: Feature Matrix Construction

Concatenate CLR-transformed OTU features with the corrected_gai feature to form the final feature matrix `X`.
This hybrid approach combines:
- **Microbiome signal**: 900+ CLR-normalized OTU features capturing bacterial composition
- **Aging signal**: 1 continuous feature representing the "aging anomaly"

In [ ]:
print("=" * 70)
print("FEATURE MATRIX CONSTRUCTION")
print("=" * 70)

# Create a DataFrame with corrected_gai
gai_df = pd.DataFrame({
    'corrected_gai': corrected_gai
})

# Concatenate CLR-transformed OTUs with corrected_gai
X = pd.concat([otu_clr, gai_df], axis=1)

print(f"\nFeature matrix shape: {X.shape}")
print(f"  - CLR-transformed OTU features: {otu_clr.shape[1]}")
print(f"  - Gut Age Index feature: 1")
print(f"  - Total features: {X.shape[1]}")

print(f"\nFeature columns (last 10):")
print(X.columns.tolist()[-10:])

print(f"\nFeature matrix statistics:")
print(f"  Data type: {X.dtypes.unique()}")
print(f"  Missing values: {X.isnull().sum().sum()}")

print(f"\nFeature matrix head (first 5 samples, last 5 features):")
print(X.iloc[:5, -5:])

# Verify alignment with target
assert len(X) == len(y), "Feature matrix and target must have same number of samples!"
assert X.index.equals(y.index), "Feature matrix and target must share same index!"

print(f"\n✓ Feature matrix X successfully constructed")
print(f"  Shape: {X.shape} (samples × features)")
print(f"  Target y shape: {y.shape}")
print(f"  ✓ All samples properly aligned")

## Section 5: Meta-Classifier Setup with LightGBM

Initialize a LightGBMClassifier with:
- **`class_weight='balanced'`**: Critical for handling severe class imbalance (1,852 healthy vs ~180 CVD)
- **`n_jobs=-1`**: Parallel processing for faster training
- **Regularization hyperparameters**: Prevent overfitting
- **`random_state=42`**: Reproducibility

The meta-classifier learns how to combine CLR-OTU signals with GAI to make CVD predictions.

In [ ]:
print("=" * 70)
print("META-CLASSIFIER INITIALIZATION")
print("=" * 70)

# Initialize LightGBM classifier with balanced class weights
# Key hyperparameters:
# - class_weight='balanced': Automatically adjust weights inversely proportional to class frequency
# - n_jobs=-1: Use all CPU cores for parallel processing
# - num_leaves: Controls model complexity (default 31 is often good for tree boosting)
# - learning_rate: Shrinkage parameter for regularization (lower = stronger regularization)
# - max_depth: Maximum tree depth (deeper trees can overfit; 7-10 is typical)
# - random_state: For reproducibility

clf = lgb.LGBMClassifier(
    class_weight='balanced',           # Handle severe class imbalance
    n_jobs=-1,                          # Parallel processing
    num_leaves=31,                      # Tree complexity
    learning_rate=0.1,                  # Shrinkage / regularization
    max_depth=-1,                       # No depth limit (LightGBM grows leaf-wise)
    min_child_samples=20,               # Min samples per leaf
    subsample=0.8,                      # Row subsampling (bagging fraction)
    colsample_bytree=0.8,               # Column subsampling
    reg_alpha=0.0,                      # L1 regularization
    reg_lambda=1.0,                     # L2 regularization
    random_state=42,
    verbose=-1                          # Suppress training output
)

print("\nLightGBM Classifier Configuration:")
print(f"  Class weight: balanced")
print(f"  Parallel jobs: all available cores")
print(f"  Number of leaves: 31")
print(f"  Learning rate: 0.1")
print(f"  Subsample fraction: 0.8")
print(f"  Column subsample fraction: 0.8")
print(f"  Random state: 42 (reproducible)")

print("\n✓ Classifier initialized successfully")

## Section 6: Stratified Cross-Validation Evaluation

Implement rigorous evaluation with **StratifiedKFold** (5 splits) to prevent data leakage:
- Ensure same class distribution in each fold
- Train on fold-specific training set
- Evaluate on hold-out validation set
- Repeat for all 5 folds and aggregate results

This prevents the model from seeing validation data during training.

In [ ]:
print("=" * 70)
print("STRATIFIED K-FOLD CROSS-VALIDATION (5 FOLDS)")
print("=" * 70)

# Initialize StratifiedKFold with 5 splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage for results
fold_results = []
all_y_val = []
all_y_pred_proba = []

print("\nStarting cross-validation evaluation...\n")

# Loop through each fold
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Fold {fold_idx}/5:")
    
    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Print class distribution in this fold
    n_healthy_train = (y_train == 0).sum()
    n_cvd_train = (y_train == 1).sum()
    n_healthy_val = (y_val == 0).sum()
    n_cvd_val = (y_val == 1).sum()
    
    print(f"  Train set: {len(X_train)} samples ({n_healthy_train} healthy, {n_cvd_train} CVD)")
    print(f"  Val set:   {len(X_val)} samples ({n_healthy_val} healthy, {n_cvd_val} CVD)")
    
    # Fit classifier on training fold
    clf.fit(X_train, y_train)
    
    # Predict on validation fold
    y_pred = clf.predict(X_val)
    y_pred_proba = clf.predict_proba(X_val)[:, 1]  # Probability of CVD class
    
    # Calculate metrics
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    
    print(f"  → Balanced Accuracy: {balanced_acc:.4f}")
    print(f"  → AUC-ROC:           {auc_roc:.4f}")
    
    # Store results
    fold_results.append({
        'fold': fold_idx,
        'balanced_accuracy': balanced_acc,
        'auc_roc': auc_roc
    })
    
    all_y_val.extend(y_val.values)
    all_y_pred_proba.extend(y_pred_proba)
    
    print()

# Convert to DataFrame for analysis
results_df = pd.DataFrame(fold_results)

print("\n" + "=" * 70)
print("CROSS-VALIDATION RESULTS SUMMARY")
print("=" * 70)

print(f"\nPer-fold results:")
print(results_df.to_string(index=False))

print(f"\nAggregate metrics across 5 folds:")
print(f"  Balanced Accuracy: {results_df['balanced_accuracy'].mean():.4f} ± {results_df['balanced_accuracy'].std():.4f}")
print(f"  AUC-ROC:           {results_df['auc_roc'].mean():.4f} ± {results_df['auc_roc'].std():.4f}")

print(f"\nMinimum/Maximum values:")
print(f"  Balanced Accuracy: min={results_df['balanced_accuracy'].min():.4f}, max={results_df['balanced_accuracy'].max():.4f}")
print(f"  AUC-ROC:           min={results_df['auc_roc'].min():.4f}, max={results_df['auc_roc'].max():.4f}")

# Check if targets met
mean_balanced_acc = results_df['balanced_accuracy'].mean()
mean_auc = results_df['auc_roc'].mean()

if mean_balanced_acc > 0.70 and mean_auc > 0.75:
    print(f"\n✓ TARGET METRICS MET!")
elif mean_balanced_acc > 0.70:
    print(f"\n⚠ Balanced Accuracy target (>70%) met, but AUC-ROC target (>75%) not met")
elif mean_auc > 0.75:
    print(f"\n⚠ AUC-ROC target (>75%) met, but Balanced Accuracy target (>70%) not met")
else:
    print(f"\n⚠ Target metrics not yet met. Further model tuning may be needed.")

## Section 7: Performance Metrics Visualization

Create visual summaries of cross-validation results:
1. Bar plot comparing Balanced Accuracy and AUC-ROC across folds
2. ROC curve aggregating predictions from all folds

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Per-fold metrics comparison
ax1 = axes[0]
x_pos = np.arange(len(results_df))
width = 0.35

bars1 = ax1.bar(x_pos - width/2, results_df['balanced_accuracy'], width, 
                label='Balanced Accuracy', alpha=0.8, color='steelblue')
bars2 = ax1.bar(x_pos + width/2, results_df['auc_roc'], width, 
                label='AUC-ROC', alpha=0.8, color='coral')

# Add horizontal lines for means
ax1.axhline(y=results_df['balanced_accuracy'].mean(), color='steelblue', 
            linestyle='--', alpha=0.6, linewidth=2, label=f"BA Mean: {results_df['balanced_accuracy'].mean():.3f}")
ax1.axhline(y=results_df['auc_roc'].mean(), color='coral', 
            linestyle='--', alpha=0.6, linewidth=2, label=f"AUC Mean: {results_df['auc_roc'].mean():.3f}")

ax1.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
ax1.set_title('Cross-Validation: Per-Fold Performance', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f"Fold {i}" for i in range(1, len(results_df)+1)])
ax1.set_ylim([0.5, 1.0])
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3, axis='y')

# Plot 2: ROC curve (aggregated from all folds)
ax2 = axes[1]
fpr, tpr, thresholds = roc_curve(all_y_val, all_y_pred_proba)
auc_overall = roc_auc_score(all_y_val, all_y_pred_proba)

ax2.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'ROC Curve (AUC = {auc_overall:.4f})')
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1, label='Random Classifier')
ax2.fill_between(fpr, tpr, alpha=0.2, color='blue')

ax2.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax2.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax2.set_title('ROC Curve: Aggregated CV Predictions', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='lower right')
ax2.grid(alpha=0.3)
ax2.set_xlim([-0.02, 1.02])
ax2.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig('cv_performance_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved to 'cv_performance_summary.png'")

## Section 8: SHAP Interpretability Analysis

Use **SHapley Additive exPlanations** (SHAP) to interpret the model:
1. Fit LightGBM on the **entire dataset** (for final interpretability analysis)
2. Initialize `shap.TreeExplainer` with the fitted model
3. Generate SHAP values for all samples
4. Create summary plots to visualize feature importance

**Key Question**: Does `corrected_gai` rank prominently alongside individual CLR-transformed OTU features?

In [ ]:
print("=" * 70)
print("SHAP INTERPRETABILITY ANALYSIS")
print("=" * 70)

# Re-fit the classifier on the entire dataset for final interpretability
print("\nFitting LightGBM on entire dataset for SHAP analysis...")
final_clf = lgb.LGBMClassifier(
    class_weight='balanced',
    n_jobs=-1,
    num_leaves=31,
    learning_rate=0.1,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)

final_clf.fit(X, y)
print("✓ Model fitted on full dataset")

# Initialize SHAP TreeExplainer
print("\nInitializing SHAP TreeExplainer...")
explainer = shap.TreeExplainer(final_clf)

# Calculate SHAP values for all samples
print("Computing SHAP values for all samples (this may take a moment)...")
shap_values = explainer.shap_values(X)

# Note: For binary classification, shap_values is a list with 2 elements
# shap_values[1] corresponds to the positive class (CVD)
if isinstance(shap_values, list):
    shap_values_cvd = shap_values[1]
    print(f"✓ SHAP values computed (shape: {shap_values_cvd.shape})")
else:
    shap_values_cvd = shap_values
    print(f"✓ SHAP values computed (shape: {shap_values_cvd.shape})")

# Create SHAP summary plot (feature importance)
print("\nGenerating SHAP summary plot...")

fig, ax = plt.subplots(figsize=(12, 8))

# Create summary plot
shap.summary_plot(shap_values_cvd, X, plot_type="bar", show=False)

plt.title('SHAP Feature Importance: CVD Prediction\n(Top 30 features)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Mean |SHAP value|', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ SHAP bar plot saved to 'shap_summary_bar.png'")

# Create SHAP beeswarm summary plot (shows distribution of SHAP values)
print("\nGenerating SHAP beeswarm plot...")

fig, ax = plt.subplots(figsize=(12, 10))

# Beeswarm plot
shap.summary_plot(shap_values_cvd, X, plot_type="violin", show=False)

plt.title('SHAP Feature Impact: CVD Prediction\n(Color: Feature value, Position: SHAP value)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('SHAP value (impact on model output)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_violin.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ SHAP violin plot saved to 'shap_summary_violin.png'")

# Identify the rank and importance of corrected_gai
print("\n" + "=" * 70)
print("CORRECTED_GAI FEATURE ANALYSIS")
print("=" * 70)

# Calculate mean absolute SHAP values for each feature
mean_abs_shap = np.abs(shap_values_cvd).mean(axis=0)

# Create a ranking dataframe
feature_ranking = pd.DataFrame({
    'feature': X.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

# Find rank and stats for corrected_gai
gai_rank_idx = feature_ranking[feature_ranking['feature'] == 'corrected_gai'].index
if len(gai_rank_idx) > 0:
    gai_rank = gai_rank_idx[0] + 1  # 1-indexed
    gai_importance = feature_ranking.iloc[gai_rank_idx[0]]['mean_abs_shap']
    
    print(f"\nGAI Feature Ranking:")
    print(f"  Rank: #{gai_rank} out of {len(feature_ranking)} features")
    print(f"  Mean |SHAP value|: {gai_importance:.6f}")
    print(f"  Percentile: {100 * (1 - gai_rank/len(feature_ranking)):.1f}%")
    
    print(f"\nTop 10 most important features:")
    top_10 = feature_ranking.head(10)
    for idx, row in top_10.iterrows():
        marker = " ← corrected_gai" if row['feature'] == 'corrected_gai' else ""
        print(f"  {idx+1:2d}. {row['feature']:20s} : {row['mean_abs_shap']:.6f}{marker}")
    
    print(f"\nInterpretation:")
    if gai_rank <= 10:
        print(f"  ✓ corrected_gai ranks in TOP 10 features (excellent signal!)")
    elif gai_rank <= 50:
        print(f"  ✓ corrected_gai ranks in TOP 50 features (good signal)")
    else:
        print(f"  ⚠ corrected_gai ranks outside top 50 (consider model tuning)")
else:
    print("⚠ corrected_gai feature not found in model!")

## Summary & Next Steps

### Key Results
- **Cross-validation evaluation** on hold-out folds ensures no data leakage
- **Balanced Accuracy & AUC-ROC** capture model performance on imbalanced data
- **SHAP analysis** reveals which features drive CVD predictions

In [ ]:
print("\n" + "=" * 70)
print("HYBRID STACKING WORKFLOW: EXECUTION COMPLETE")
print("=" * 70)

print(f"""
SUMMARY OF RESULTS:
───────────────────────────────────────────────────────────────────────

Dataset:               {DATASET_NAME}
Total samples:         {len(X)}
  Healthy controls:    {(y == 0).sum()}
  CVD patients:        {(y == 1).sum()}

Feature Matrix:
  CLR-transformed OTUs: {otu_clr.shape[1]}
  Corrected GAI:        1
  Total features:       {X.shape[1]}

Cross-Validation (5-fold StratifiedKFold):
  Mean Balanced Accuracy: {results_df['balanced_accuracy'].mean():.4f} ± {results_df['balanced_accuracy'].std():.4f}
  Mean AUC-ROC:           {results_df['auc_roc'].mean():.4f} ± {results_df['auc_roc'].std():.4f}

SHAP Feature Importance:
  Corrected_gai rank:     #{gai_rank} out of {len(feature_ranking)}
  Mean |SHAP value|:      {gai_importance:.6f}

───────────────────────────────────────────────────────────────────────

NEXT STEPS FOR IMPROVEMENT:
1. Hyperparameter tuning:
   - Use GridSearchCV or Bayesian optimization on LightGBM parameters
   - Test num_leaves, learning_rate, max_depth variations

2. Feature selection:
   - Identify and retain top 50-100 most important OTUs (by SHAP values)
   - Reduce feature dimensionality to improve generalization

3. Cross-cohort validation:
   - Train on AGP, test on GGMP (or vice versa)
   - Verify model generalizes across populations

4. Ensemble methods:
   - Combine LightGBM with XGBoost or CatBoost
   - Stack predictions for improved robustness

5. Data augmentation:
   - Include metabolic pathway features (PICRUSt2)
   - Add functional annotations (KEGG, eggNOG)
   - Combine with clinical features if available

6. Threshold optimization:
   - Adjust decision threshold based on clinical requirements
   - Optimize for sensitivity vs. specificity trade-off

───────────────────────────────────────────────────────────────────────
""")

print("✓ Notebook execution complete!")
print("✓ SHAP plots saved: shap_summary_bar.png, shap_summary_violin.png")
print("✓ CV performance plot saved: cv_performance_summary.png")